In [36]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [37]:
# All files under the input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [38]:
seed=42
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)

# Global constants
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

In [39]:
def calculate_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argsort(logits, axis=-1)[:, ::-1] # Sort in descending order

    top1_preds = preds[:, 0]
    accuracy = accuracy_score(labels, top1_preds)
    f1 = f1_score(labels, top1_preds, average="macro")

    map3_score = 0.0
    for i in range(len(labels)):
        true_label = labels[i]
        for rank in range(3):
            if preds[i, rank] == true_label:
                map3_score += 1.0 / (rank + 1)
                break
    map3_score /= len(labels)
    
    return {
        "accuracy": accuracy,
        "f1_score": f1,
        "map@3": map3_score
    }
    # It extracts logits (raw model predictions) and labels, and
    # calculates standard Accuracy, Macro F1-score, and iteratively calculates the MAP@3 score.

In [40]:
df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Step 1: Case normalization - Convert all text to lowercase
df['prompt'] = df['prompt'].apply(lambda x: x.lower())
test_df['prompt'] = df['prompt'].apply(lambda x: x.lower())

# Display example of case normalization effect
print("\nCase Normalization Example:")
original_text = df['prompt'].iloc[0][:100]
normalized_text = df['prompt'].iloc[0][:100]
print(f"Original: {original_text}...")
print(f"Normalized: {normalized_text}...")


Case Normalization Example:
Original: pick the best possible answer: what is martin heidegger's view on the relationship between time and ...
Normalized: pick the best possible answer: what is martin heidegger's view on the relationship between time and ...


In [41]:
count = (~df['prompt'].str.contains(r'[?:]', na=False)).sum()
count

np.int64(0)

In [42]:
# Step 2: Punctuation removal
from string import punctuation
print("\nPunctuation characters to remove:")
print(punctuation)

# Remove all punctuation characters from the normalized text
print("\nApplying punctuation removal...")
df['prompt'] = df['prompt'].apply(
    lambda x: ''.join(c for c in x if c not in punctuation or c in '?:')
)

# Display example of punctuation removal effect
print("\nPunctuation Removal Example:")
normalized_with_punct = df['prompt'].iloc[0][:100]
cleaned_text = df['prompt'].iloc[0][:100]
print(f"Before: {normalized_with_punct}...")
print(f"After: {cleaned_text}...")


Punctuation characters to remove:
!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

Applying punctuation removal...

Punctuation Removal Example:
Before: pick the best possible answer: what is martin heideggers view on the relationship between time and h...
After: pick the best possible answer: what is martin heideggers view on the relationship between time and h...


In [43]:
df.head(6)

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer: what is martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,what is acceleratorbased lightion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,determine the correct option: what is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,select the most accurate option: what is marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,identify the correct statement: what is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
5,6,identify the correct statement: what is the ef...,"An electric field, precisely aligned with the ...","A magnetic field, randomly aligned with the sp...","A magnetic field, precisely aligned with the s...","A gravitational field, randomly aligned with t...","A gravitational field, precisely aligned with ...",C


In [44]:
# Stratified split to ensure answer distributions match
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['answer'])

# To build a robust vocabulary, we compile all prompts and options from the training set
train_texts = train_df['prompt'].tolist()
for opt in ['A', 'B', 'C', 'D', 'E']:
    train_texts.extend(train_df[opt].tolist())

# Initialize vectorizer
vectorizer = TfidfVectorizer(max_df=0.95) 
# Ignores words that appear in over 95% of the text (useless for discrimination))

# Fit the model to learn the vocabulary and IDF weights from the training data
vectorizer.fit(train_texts) 

TfidfVectorizer(max_df=0.95)

In [45]:
# EVALUATE ON VALIDATION DATA
val_probs = []
val_labels = []

# Process row by row for validation
for idx, row in val_df.iterrows():
    prompt = str(row['prompt'])
    options = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
    
    # Transform text into TF-IDF vectors
    prompt_vec = vectorizer.transform([prompt])
    options_vec = vectorizer.transform(options)
    
    # Calculate cosine similarity between the prompt and each of the 5 options
    sims = cosine_similarity(prompt_vec, options_vec)[0] 
    
    val_probs.append(sims)
    val_labels.append(LABEL_MAP[row['answer']])

# Convert to numpy arrays for metric calculation
val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

In [46]:
# Model Evaluation
# Sort predictions in descending order to rank the highest similarities first
preds_ranked = np.argsort(val_probs, axis=-1)[:, ::-1]

top1_preds = preds_ranked[:, 0]
val_accuracy = accuracy_score(val_labels, top1_preds)
val_f1 = f1_score(val_labels, top1_preds, average='macro')

# MAP@3 Metric
map3_score = 0.0
for i in range(len(val_labels)):
    true_label = val_labels[i]
    for rank in range(3):
        if preds_ranked[i, rank] == true_label:
            map3_score += 1.0 / (rank + 1)
            break
map3_score /= len(val_labels)

print("\n--- TF-IDF Validation Performance ---")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation F1 Score: {val_f1:.4f}")
print(f"Validation MAP@3:    {map3_score:.4f}")


--- TF-IDF Validation Performance ---
Validation Accuracy: 0.1275
Validation F1 Score: 0.1268
Validation MAP@3:    0.2583
